# Separação dos dados em treino, validação e teste

## Estratégia

O objetivo é prever as kills da próxima partida usando o histórico do jogador. Por isso a divisão é feita **por data**, e não por sorteio: o treino usa as partidas mais antigas, a validação as seguintes e o teste as mais recentes. Assim, nenhuma partida futura é usada para treinar um modelo avaliado em partidas anteriores.

## Proporções

| Conjunto | Proporção | Uso |
|---|---|---|
| Treino | 70% | O modelo aprende |
| Validação | 15% | Comparar e ajustar modelos |
| Teste | 15% | Avaliação final, usada só depois das decisões |

Justificativa: com mais de 170 mil linhas, 15% (cerca de 26 mil) já é suficiente para avaliar com estabilidade, e 70% dá bastante dado para o modelo aprender.

## Dados de entrada

A divisão é aplicada ao dataset final (`src/data/dataset_final.py`), que já passou pelas etapas de preparação:

1. Limpeza e validação dos dados (`clean_dataset`).
2. Criação das informações do histórico (`criar_features_historico`). Elas usam só partidas anteriores, então calculá-las antes da divisão não vaza informação.
3. Manter só linhas com pelo menos 5 partidas anteriores (`filtrar_minimo_partidas`).
4. Remover linhas com valores ausentes nas entradas ou no alvo.

Depois disso, o dataset é dividido por data com `dividir_por_tempo`.


In [1]:
import sys
sys.path.append("..")

from pathlib import Path

from src.data.dataset_final import carregar_dataset_final, gerar_dataset_final
from src.preprocessing.separacao_dados import (
    dividir_por_tempo,
    resumo_separacao,
    verificar_ordem_temporal,
)

ENTRADA = Path("../data/raw/tb_lobby_stats_player.csv")
CAMINHO_FINAL = Path("../data/processed/tb_lobby_stats_player_final.csv")

# Gera o dataset final se ele ainda não existe nesta máquina
if not CAMINHO_FINAL.exists():
    gerar_dataset_final(ENTRADA, CAMINHO_FINAL)

df = carregar_dataset_final(CAMINHO_FINAL)

treino, validacao, teste = dividir_por_tempo(df)
verificar_ordem_temporal(treino, validacao, teste)

print("Linhas do dataset final:", len(df))
display(resumo_separacao(treino, validacao, teste))

Linhas do dataset final: 172773


,Linhas,Porcentagem (%),Primeira data,Última data,Jogadores
Treino,120941,70.0,2021-09-14 19:07:41,2022-01-09 00:05:31,1608
Validação,25916,15.0,2022-01-09 00:06:07,2022-01-26 03:52:44,1549
Teste,25916,15.0,2022-01-26 03:53:43,2022-02-11 13:27:48,1671


In [2]:
print("Soma dos conjuntos:", len(treino) + len(validacao) + len(teste))
print("Jogadores do teste que não aparecem no treino:",
      len(set(teste['idPlayer']) - set(treino['idPlayer'])), "de", teste['idPlayer'].nunique())

print("Média de kills (treino, validação, teste):",
      round(treino['qtKill'].mean(), 2), round(validacao['qtKill'].mean(), 2), round(teste['qtKill'].mean(), 2))


Soma dos conjuntos: 172773
Jogadores do teste que não aparecem no treino: 352 de 1671
Média de kills (treino, validação, teste): 19.2 19.11 19.1


## Resultados

- Os três conjuntos somam 172.773 linhas: treino 120.941 (70,0%), validação 25.916 (15,0%) e teste 25.916 (15,0%).
- Os períodos não se sobrepõem: o treino vai de 14/09/2021 a 09/01/2022, a validação de 09/01/2022 a 26/01/2022 e o teste de 26/01/2022 a 11/02/2022.
- A média de kills é parecida nos três conjuntos (19,20, 19,11 e 19,10), então a divisão não criou um conjunto muito diferente dos outros.
- 352 dos 1.671 jogadores do teste (21%) não têm nenhuma linha no treino. Isso não é problema, porque o `idPlayer` não é usado como entrada: o modelo olha o histórico do jogador, e não quem ele é.
- O dataset final tem 90 linhas a menos que as 172.863 que sobram depois do filtro de 5 partidas anteriores. A coluna `qtHitHeadshot` tem valores nulos em 10 jogadores com muito histórico, e a etapa que remove registros incompletos descarta essas linhas. O efeito no erro da baseline é desprezível (MAE na validação de 5,4115 com 172.863 linhas e 5,4109 com 172.773).
